In [16]:
!curl -A "Mozilla/5.0" -L -o /content/development.mp4 https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5889k  100 5889k    0     0  16.3M      0 --:--:-- --:--:-- --:--:-- 16.3M


In [17]:
!pip install -q ultralytics

In [18]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import importlib.util
import subprocess
import sys

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

In [19]:
missing = [
    name for name in ("onnx", "onnxruntime")
    if importlib.util.find_spec(name) is None
]

if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

In [20]:
VIDEO_PATH = "/content/development.mp4"
MODEL_PATH = "yolo26n.pt"
TRACKER_PATH = "bytetrack.yaml"

START_FRAME = 500
N_FRAMES = 100
WARMUP_RUNS = 5

PREDICT_ARGS = dict(
    imgsz=640,
    rect=False,
    classes=[0, 1, 2],
    conf=0.10,
    iou=0.70,
    verbose=False,
    save=False
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "Unavailable"
)
print(f"Measured source frames: {START_FRAME}–{START_FRAME + N_FRAMES - 1}")

GPU: Tesla T4
Measured source frames: 500–599


**Export ONNX**

In [21]:
ONNX_PATH = YOLO(MODEL_PATH).export(
    format="onnx",
    imgsz=640,
    batch=1,
    dynamic=False,
    simplify=False,
    opset=17,
    device="cpu"
)

ONNX_PATH = str(ONNX_PATH)
print("Exported:", ONNX_PATH)

Ultralytics 8.4.158 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

ONNX: starting export with onnx 1.23.0 opset 17...
ONNX: export success ✅ 0.9s, saved as 'yolo26n.onnx' (9.4 MB)

Export complete (1.4s)
Results saved to /content/yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
Exported: yolo26n.onnx


**Benchmark function**

In [22]:
def benchmark(label, weights, device):
    model = YOLO(weights, task="detect")
    args = {**PREDICT_ARGS, "device":device}
    use_cuda = device != "cpu"
    
    def sync():
        if use_cuda:
            torch.cuda.synchronize(device)
        
    cap = cv2.VideoCapture(VIDEO_PATH)
    writer = None
    
    try:
        source_fps = cap.get(cv2.CAP_PROP_FPS)
        cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME - 1)
        
        ok, warm_frame = cap.read()
        height, width = warm_frame.shape[:2]
        
        for _ in range(WARMUP_RUNS):
            model.predict(warm_frame, **args)
        
        model.track(
            warm_frame, 
            persist=True,
            tracker=TRACKER_PATH,
            **args
        )
        sync()
        
        if Path(weights).suffix.lower() == ".onnx":
            providers = model.predictor.model.session.get_providers()
        
        timings = []
        with TemporaryDirectory() as temp_dir:
            writer = cv2.VideoWriter(
                str(Path(temp_dir) / "benchmark.mp4"),
                cv2.VideoWriter_fourcc(*"mp4v"),
                source_fps,
                (width, height)
            )
            run_start = perf_counter()

            for _ in range(N_FRAMES):
                t0 = perf_counter()
                ok, frame = cap.read()
                t1 = perf_counter()
                
                result = model.track(
                    frame,
                    persist=True,
                    tracker=TRACKER_PATH,
                    **args,
                )[0]
                
                sync()
                t2 = perf_counter()

                annotated = result.plot()
                sync()
                t3 = perf_counter()
                
                writer.write(annotated)
                t4 = perf_counter()
                
                timings.append([
                    (t1 - t0) * 1000,
                    (t2 - t1) * 1000,
                    (t3 - t2) * 1000,
                    (t4 - t3) * 1000,
                    (t4 - t0) * 1000,
                ])

            flush_start = perf_counter()
            writer.release()
            writer = None
            flush_ms = (perf_counter() - flush_start) * 1000
            
            total_seconds = perf_counter() - run_start
            
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    
    times = np.asarray(timings)
    stage_means = times[:, :4].mean(axis=0)
    pipeline_fps = len(times) / total_seconds
    
    return {
        "backend": label,
        "frames": len(times),
        "decode_ms": stage_means[0],
        "detect_track_ms": stage_means[1],
        "draw_ms": stage_means[2],
        "write_ms": stage_means[3],
        "mean_frame_ms": times[:, 4].mean(),
        "p95_frame_ms": np.percentile(times[:, 4], 95),
        "flush_ms": flush_ms,
        "pipeline_fps": pipeline_fps,
        "source_fps": source_fps,
        "real_time_factor": pipeline_fps / source_fps
    }

In [23]:
runs = [
    ("PyTorch_CPU", MODEL_PATH, "cpu"),
    ("ONNX_CPU", ONNX_PATH, "cpu"),
]

if torch.cuda.is_available():
    runs.append(("PyTorch_GPU", MODEL_PATH, 0))

rows = []
for label, weights, device in runs:
    print("Running:", label)
    rows.append(benchmark(label, weights, device))
    

benchmark_df = pd.DataFrame(rows).set_index("backend")
display(benchmark_df.round(3))

Running: PyTorch_CPU
Running: ONNX_CPU
Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Running: PyTorch_GPU


,frames,decode_ms,detect_track_ms,draw_ms,write_ms,mean_frame_ms,p95_frame_ms,flush_ms,pipeline_fps,source_fps,real_time_factor
backend,,,,,,,,,,,
PyTorch_CPU,100,0.837,121.470,0.760,2.754,125.820,173.408,0.284,7.947,12.0,0.662
ONNX_CPU,100,0.830,93.716,0.787,2.868,98.200,130.030,0.236,10.183,12.0,0.849
PyTorch_GPU,100,0.594,14.437,0.571,2.206,17.808,20.860,0.228,56.138,12.0,4.678
